# 03 · Sorting cuboids between wells

Moving cuboids from the wells of one plate into the wells of another, in an
order decided by a table of per-well values measured elsewhere — luminescence,
pixel intensity, whatever the other instrument produced.

This is a standalone procedure, not part of the picking pipeline: no vision, no
state machine, no `PickingSession`. There is a table, an order, and an operator
who looks at the tip after every pickup.

**What it does**

1. Brings the robot up and loads both plates.
2. Helps measure the well-centre offset and the top of the plate, and derives
   the well bottom from a depth you supply.
3. Reads the table of values and builds a `source well -> destination well`
   assignment.
4. Runs a semi-automatic loop: the robot aspirates, parks above the well and
   waits; you look at the tip and press a key.
5. Writes every event to CSV, so what went where is on disk afterwards.

**Order of use.** Sections 1 to 7 in order. Sections 3 and 4 (offsets and
heights) can be skipped once the numbers are known — they are stored on disk
and reloaded in section 2.

**Before starting:** a tip is on the pipette, both plates are in their slots,
and the robot has been homed at least once since power-on.

## 0. Imports

`bench_setup` fixes `MICROPICK_ROOT` and re-exports the common names, so the
imports here are only what this notebook adds.

In [ ]:
import csv
import json
import re
from pathlib import Path

import pandas as pd

from bench_setup import *                       # noqa: F401,F403
from micropick.hardware.protocols import move_relative, move_to, require_ok, xyz

pd.set_option("display.width", 200)
print(paths.describe())

## 1. Robot and plates

`connect_robot()` creates the run and loads the pipette; nothing moves without
both. Custom labware definitions have to be uploaded into **every** run; stock
`opentrons` definitions do not.

In [ ]:
PROFILE = "lab_main"

profile = load_profile(PROFILE)
print("profile:", profile.meta.name)

In [ ]:
openapi = ot2_api.OpentronsAPI()
openapi.add_slot_offsets([5, 8, 9], (0, 0, 64.2))

In [ ]:
# Use to restore labware and general run information after the notebook crashes
r = openapi.get_run_info()

In [ ]:
# Custom definitions available locally. Stock ones are not listed here, but
# resolve_definition finds them by load name.
print("\n".join(f"  {n}" for n in sorted(local_definitions())) or "  (none)")

In [ ]:
r = openapi.load_labware(DEST_PLATE, 4, namespace='opentrons',verbose=True)

In [ ]:
# --- Plates ---------------------------------------------------------------
SOURCE_SLOT = 1
DEST_SLOT   = 1              # may be the same as SOURCE_SLOT

SOURCE_PLATE = DEST_PLATE = "corning_384_wellplate_112ul_flat"
# DEST_PLATE   = "corning_6_wellplate_16.8ml_flat"

# SOURCE_PLATE = "corning_96_wellplate_360ul_flat"     # load name
# DEST_PLATE   = "corning_96_wellplate_360ul_flat"
src_def  = resolve_definition(SOURCE_PLATE)
dest_def = resolve_definition(DEST_PLATE)
print("source:     ", src_def)
print("destination:", dest_def)

In [ ]:
# Upload custom definitions into this run. An empty labware/ is not an error
# when both plates are stock.
try:
    labware.ensure_definitions(openapi, verbose=True)
except LabwareError as exc:
    print("nothing to upload:", exc)

labware.load_labware(openapi, SOURCE_PLATE, SOURCE_SLOT)
if DEST_SLOT != SOURCE_SLOT:
    labware.load_labware(openapi, DEST_PLATE, DEST_SLOT)

src_lw  = openapi.labware_dct[str(SOURCE_SLOT)]
dest_lw = openapi.labware_dct[str(DEST_SLOT)]
print(f"slot {SOURCE_SLOT}: {src_lw}\nslot {DEST_SLOT}: {dest_lw}")

In [ ]:
# What the robot actually believes is loaded. Worth reading against the deck
# before the first move.
for slot, lw in sorted(loaded_labware(openapi).items()):
    print(f"  slot {slot}: {lw.load_name} v{lw.version} ({lw.namespace})")

## 2. Measured setup

Everything measured on the bench — the two well-centre offsets, the top of the
plate, the well depth — lives in one `setup` dictionary that is written to disk
on every change and read directly by the motion helpers.

Nothing is copied between cells by hand. Re-measuring an offset in section 3
changes what section 7 uses, immediately and without a variable to remember.

The file is per-run-of-the-experiment, not per-installation: it belongs to
`outputs/`, not to the profile, because it describes how *this* plate is
sitting in *this* slot today.

In [ ]:
SETUP_PATH = paths.outputs_dir() / "sort_setup.json"

DEFAULT_SETUP = {
    "source_offset":  [0.0, 0.0],   # mm, xy, added to every source well
    "dest_offset":    [0.0, 0.0],   # mm, xy, added to every destination well
    "plate_top_z":    None,         # mm, absolute, top rim of the source plate
    "well_depth_mm":  None,         # mm, supplied by you (see section 4)
    "bottom_z":       None,         # mm, absolute; set directly to bypass the
                                    # top-minus-depth calculation
}


def load_setup(path=None) -> dict:
    """Read the saved setup, filling anything absent from the defaults."""
    path = Path(path or SETUP_PATH)
    data = dict(DEFAULT_SETUP)
    if path.exists():
        data.update(json.loads(path.read_text(encoding="utf-8")))
        print(f"loaded {path}")
    else:
        print(f"no saved setup at {path}; starting from defaults")
    return data


def save_setup(data=None, path=None) -> None:
    path = Path(path or SETUP_PATH)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data if data is not None else setup, indent=2)
                    + "\n", encoding="utf-8")


def source_offset() -> tuple:
    return tuple(setup["source_offset"])


def dest_offset() -> tuple:
    return tuple(setup["dest_offset"])


def bottom_z() -> float:
    """Absolute Z of the source well bottom.

    Derived on every call rather than stored in a variable, so a re-measured
    plate top or a corrected depth cannot leave a stale number behind.
    """
    if setup.get("bottom_z") is not None:
        return float(setup["bottom_z"])
    top, depth = setup.get("plate_top_z"), setup.get("well_depth_mm")
    if top is None or depth is None:
        raise RuntimeError(
            "no well bottom yet: measure the plate top and set the well depth "
            "(section 4), or set setup['bottom_z'] directly")
    return float(top) - float(depth)


def show_setup() -> None:
    print(f"source offset  : {source_offset()}")
    print(f"dest offset    : {dest_offset()}")
    print(f"plate top      : {setup['plate_top_z']}")
    print(f"well depth     : {setup['well_depth_mm']}")
    print(f"bottom override: {setup['bottom_z']}")
    try:
        print(f"-> well bottom : {bottom_z():.2f} mm")
    except RuntimeError as exc:
        print(f"-> well bottom : unavailable ({exc})")


setup = load_setup()
show_setup()

## 3. Well-centre offset

The robot's idea of a well centre comes from the labware definition, and the
real centre is a fraction of a millimetre away from it: how the plate sits in
the slot, and the tip's own offset, both contribute. The correction is measured
once per plate and used for every well of it.

Two ways to get there, both ending in `setup`:

* **by trial** — call `check_offset` with a guess, look, adjust, repeat (the way
  this was done before);
* **by driving there** — park at the nominal centre, walk the tip over with
  `nudge()` or `jog()`, then `capture_offset()` reads the correction off the
  pose.

A corner well (`A1`, `H1`) shows a badly seated plate more clearly than a
central one.

In [ ]:
CHECK_Z = 0.0              # mm above the well top while checking
LIMITS  = Limits(x=(0, 380), y=(0, 350), z=(0.1, 150))

_nominal = {}              # "source"/"dest" -> pose at the well's nominal top


def _plate(which):
    if which == "source":
        return src_lw
    if which == "dest":
        return dest_lw
    raise ValueError(f"which: 'source' or 'dest', got {which!r}")


def well_top(which, well, offset=(0.0, 0.0), z=CHECK_Z, direct=False):
    """Park above a well top with an xy offset. Returns the pose reached."""
    require_ok(openapi.move_to_well(_plate(which), well, well_location="top",
                                    offset=(offset[0], offset[1], z),
                                    force_direct=direct),
               f"move to {well} top")
    pose = xyz(openapi)
    print(f"{which} {well} top{tuple(round(v, 2) for v in offset)} -> "
          f"({pose[0]:.2f}, {pose[1]:.2f}, {pose[2]:.2f})")
    return pose


def nudge(axis, mm):
    """A step on one axis, verified against the pose read back."""
    pose = move_relative(openapi, axis, mm)
    print(f"({pose[0]:.2f}, {pose[1]:.2f}, {pose[2]:.2f})")
    return tuple(pose)


def jog(step=0.1, title=""):
    """Jog in its own window: arrows for xy, q/e for z, +/- for step size,
    Enter to finish. The window must have focus. With no camera it is blank,
    but the keys still drive the robot."""
    ctrl = JogController(openapi, limits=LIMITS, step=step)
    pos = jog_in_window(ctrl, camera=None, title=title or "jog")
    print("stopped at", tuple(round(v, 2) for v in pos))
    return pos


def nominal_top(which, well, z=CHECK_Z):
    """Park at the well's nominal centre (zero offset) and remember the pose,
    so capture_offset has something to measure against."""
    pose = well_top(which, well, offset=(0.0, 0.0), z=z)
    _nominal[which] = pose
    return pose


def capture_offset(which):
    """Store the current pose's departure from the last nominal centre."""
    if which not in _nominal:
        raise RuntimeError(f"call nominal_top({which!r}, well) first")
    cur = xyz(openapi)
    ref = _nominal[which]
    setup[f"{which}_offset"] = [round(cur[0] - ref[0], 3),
                                round(cur[1] - ref[1], 3)]
    save_setup()
    print(f"{which} offset -> {setup[f'{which}_offset']}  (saved)")
    return tuple(setup[f"{which}_offset"])


def set_offset(which, dx, dy):
    """Store an offset typed in by hand."""
    setup[f"{which}_offset"] = [round(float(dx), 3), round(float(dy), 3)]
    save_setup()
    print(f"{which} offset -> {setup[f'{which}_offset']}  (saved)")
    return tuple(setup[f"{which}_offset"])


def check_offset(which, well, z=CHECK_Z):
    """Park using the offset that is *stored*, not one typed into this call —
    so what you look at is what the run will use."""
    off = source_offset() if which == "source" else dest_offset()
    return well_top(which, well, offset=off, z=z)

In [ ]:
CHECK_WELL = "H1"          # corner well used for the measurement

# --- Path A: by trial -----------------------------------------------------
# Store a guess, look at it, adjust, repeat.
set_offset("source", 0.1, -0.6)
check_offset("source", CHECK_WELL)

In [ ]:
set_offset("dest", 0.1, -0.6)

In [ ]:
# --- Path B: by driving there ---------------------------------------------
# 1) park at the nominal centre
nominal_top("source", CHECK_WELL)

In [ ]:
# 2) walk the tip onto the real centre: nudge("x", 0.1) / nudge("y", -0.1),
#    or jog(step=0.1). Repeat this cell as needed.
nudge("x", 0.1)

In [ ]:
# 3) store the correction
capture_offset("source")
check_offset("source", CHECK_WELL)

In [ ]:
# --- Destination plate ----------------------------------------------------
# Same two paths. If both plates are the same and sit identically, copy across:
#     set_offset("dest", *source_offset())
nominal_top("dest", "A1")

In [ ]:
capture_offset("dest")
show_setup()

## 4. Plate top and well bottom

The top of the plate is measured by touch: bring the tip down to the rim and
record Z. The bottom is that minus the well depth.

**The depth is yours to supply.** Not every plate here has a definition with
usable well geometry, and where a definition exists its depth is a description
of the catalogue part rather than the plate on the deck. `depth_hint` reads the
definition when it can, purely as something to check your number against.

Bring the tip down in small steps. It is long and it flexes, so the touch is
something you see, not something the robot reports.

In [ ]:
def depth_hint(defn, well="A1"):
    """Well depth from the labware definition, or None if it does not say."""
    try:
        return float(defn.data["wells"][well]["depth"])
    except (KeyError, TypeError, ValueError):
        return None


def set_well_depth(mm):
    setup["well_depth_mm"] = float(mm)
    save_setup()
    print(f"well depth -> {setup['well_depth_mm']} mm  (saved)")
    return setup["well_depth_mm"]


def capture_plate_top():
    """Record the current Z as the top rim of the source plate."""
    setup["plate_top_z"] = round(xyz(openapi)[2], 3)
    save_setup()
    print(f"plate top -> {setup['plate_top_z']} mm  (saved)")
    return setup["plate_top_z"]


def set_bottom_z(z=None):
    """Set the well bottom directly, or pass None to go back to
    plate top minus well depth."""
    setup["bottom_z"] = None if z is None else round(float(z), 3)
    save_setup()
    print(f"bottom override -> {setup['bottom_z']}  (saved)")


hint = depth_hint(src_def, CHECK_WELL)
print(f"definition says: {hint} mm" if hint is not None
      else "the definition gives no depth for this well")

In [ ]:
# The depth you actually want to use, in mm.
set_well_depth(12.5)

In [ ]:
# Park well above the rim, with the stored offset.
check_offset("source", CHECK_WELL, z=5.0)

In [ ]:
# Come down in steps until the tip meets the rim.
# nudge("z", -0.5) ... nudge("z", -0.1)
nudge("z", -0.5)

In [ ]:
# Touching. Record it; the bottom follows from the depth.
capture_plate_top()
show_setup()

move_relative(openapi, "z", 5.0)
openapi.retract_axis("leftZ")

In [ ]:
openapi.home_robot()

In [ ]:
show_setup()


In [ ]:
openapi.dispense_in_place(5)

## 5. The table of values

Three layouts are read:

* `tecan` — an i-control export as the plate reader writes it: metadata block,
  then a grid introduced by `<>` carrying the column numbers, then one row per
  row letter. Only the part of the plate that was read appears, so wells outside
  it simply have no value.
* `grid` — shaped like the plate and nothing else: row letters down the first
  column, column numbers across the header. An empty cell means an empty well.
* `long` — two columns, a well name and a value.

`.xlsx`, `.xlsm` and `.csv` all work for `grid` and `long`; `tecan` is Excel
only. Well names are checked against the plate definition, so a typo like `I13`
surfaces here rather than mid-run.

**Several reads of one plate.** A Tecan file often holds repeats meant to be
averaged. `sheet=None` takes every sheet, and every `<>` grid within a sheet, as
a separate read and averages them. Because averaging unrelated reads is silent
and ruinous, the reads are checked first: a different plate type or a different
set of wells raises rather than warns. The printed spread across reads is the
other half of that check — a median CV in the tens of percent means the reads
are not repeats of the same thing, whatever the metadata says.

In [ ]:
# --- Tecan i-control export -----------------------------------------------

_ROW_LABEL = re.compile(r"^[A-Za-z]{1,2}$")


def _tecan_meta(df, key):
    """The first value to the right of a metadata label in column A."""
    for r in range(len(df)):
        first = df.iat[r, 0]
        if isinstance(first, str) and first.strip().rstrip(":").lower() == key.lower():
            for c in range(1, df.shape[1]):
                value = df.iat[r, c]
                if not pd.isna(value):
                    return str(value).strip()
    return None


def _tecan_blocks(df):
    """Every '<>' grid on one sheet, as (values, unreadable count) pairs.

    A grid starts at the '<>' cell, takes its column numbers from that row, and
    runs down while column A holds a row label. Anything else — a blank line,
    'End Time:' — ends it, so a sheet with several reads on it yields several
    blocks.
    """
    blocks = []
    for r in range(len(df)):
        first = df.iat[r, 0]
        if not (isinstance(first, str) and first.strip() == "<>"):
            continue
        columns = {}
        for c in range(1, df.shape[1]):
            label = df.iat[r, c]
            if pd.isna(label):
                continue
            try:
                columns[c] = int(float(label))
            except (TypeError, ValueError):
                pass
        values, unreadable = {}, 0
        rr = r + 1
        while rr < len(df):
            label = df.iat[rr, 0]
            if not (isinstance(label, str) and _ROW_LABEL.match(label.strip())):
                break
            row = label.strip().upper()
            for c, column in columns.items():
                cell = df.iat[rr, c]
                if pd.isna(cell):
                    continue
                try:
                    values[f"{row}{column}"] = float(cell)
                except (TypeError, ValueError):
                    unreadable += 1          # 'OVER', '<0' and friends
            rr += 1
        if values:
            blocks.append((values, unreadable))
    return blocks


def read_tecan(path, sheet=None, agg="mean", strict=True, verbose=True,
               with_table=False):
    """Values from an i-control export, averaged over repeated reads.

    `sheet=None` reads every sheet; pass a name or a list of names to choose.
    Each sheet, and each grid within a sheet, counts as one read.
    """
    sheets = pd.read_excel(path, sheet_name=sheet, header=None)
    if isinstance(sheets, pd.DataFrame):
        sheets = {sheet if isinstance(sheet, str) else 0: sheets}

    replicates, plates, ranges, unreadable = {}, {}, {}, 0
    for name, df in sheets.items():
        blocks = _tecan_blocks(df)
        for i, (values, bad) in enumerate(blocks):
            key = str(name) if len(blocks) == 1 else f"{name}[{i + 1}]"
            replicates[key] = pd.Series(values, dtype=float)
            plates[key] = _tecan_meta(df, "Plate")
            ranges[key] = _tecan_meta(df, "Part of Plate")
            unreadable += bad

    if not replicates:
        raise ValueError(f"{Path(path).name}: no Tecan '<>' grid on any sheet")

    table = pd.DataFrame(replicates)
    distinct = {p for p in plates.values() if p}

    if verbose:
        print(f"{len(replicates)} read(s):")
        for key in replicates:
            print(f"  {key}: plate {plates[key]!r}")
            print(f"      part {ranges[key]!r}, "
                  f"{int(replicates[key].notna().sum())} wells")
        if unreadable:
            print(f"  {unreadable} cells were not numbers and were dropped")

    # Averaging reads of different things is silent and ruinous, so it raises.
    if len(distinct) > 1:
        message = f"the reads are of different plate types: {sorted(distinct)}"
        if strict:
            raise ValueError(f"{message}; pass an explicit sheet=, "
                             f"or strict=False to average anyway")
        print("warning:", message)
    if table.isna().any().any():
        missing = table.isna().sum()
        message = (f"the reads do not cover the same wells "
                   f"(missing per read: {missing[missing > 0].to_dict()})")
        if strict:
            raise ValueError(f"{message}; pass strict=False to average anyway")
        print("warning:", message)

    if agg == "mean":
        out = table.mean(axis=1)
    elif agg == "median":
        out = table.median(axis=1)
    elif agg == "first":
        out = table.iloc[:, 0]
    else:
        raise ValueError(f"agg: mean|median|first, got {agg!r}")

    # The other half of the check: metadata can agree while the numbers do not.
    if verbose and table.shape[1] > 1:
        mean = table.mean(axis=1)
        cv = (table.std(axis=1, ddof=0) / mean.where(mean != 0)).abs()
        print(f"  spread across reads: median CV {100 * cv.median():.1f}%, "
              f"worst {100 * cv.max():.1f}%")

    out = out.dropna().astype(float)
    out.name = "value"
    return (out, table) if with_table else out

In [ ]:
def read_values(path, layout="grid", sheet=0, well_col="well",
                value_col="value", agg="mean", strict=True) -> pd.Series:
    """A table of per-well values as a Series indexed by well name."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    excel = path.suffix.lower() in (".xlsx", ".xls", ".xlsm")

    if layout == "tecan":
        return read_tecan(path, sheet=sheet, agg=agg, strict=strict)

    # For the plain layouts a sheet has to be named; None would hand back a
    # dict of every sheet instead of a table.
    sheet = 0 if sheet is None else sheet

    if layout == "grid":
        df = (pd.read_excel(path, sheet_name=sheet, index_col=0) if excel
              else pd.read_csv(path, index_col=0))
        out = {}
        for row in df.index:
            label = str(row).strip().upper()
            if not label or label.startswith("UNNAMED"):
                continue
            for col in df.columns:
                m = re.match(r"^\s*(\d+)", str(col))
                if not m:
                    continue
                value = df.at[row, col]
                if pd.isna(value):
                    continue
                out[f"{label}{int(m.group(1))}"] = float(value)
    elif layout == "long":
        df = (pd.read_excel(path, sheet_name=sheet) if excel
              else pd.read_csv(path))
        missing = {well_col, value_col} - set(df.columns)
        if missing:
            raise KeyError(f"no columns {sorted(missing)} in the file; "
                           f"it has {list(df.columns)}")
        df = df[[well_col, value_col]].dropna()
        out = {str(w).strip().upper(): float(v)
               for w, v in zip(df[well_col], df[value_col])}
    else:
        raise ValueError(f"layout: 'tecan', 'grid' or 'long', got {layout!r}")

    if not out:
        raise ValueError(f"{path.name}: no values could be read")
    return pd.Series(out, dtype=float, name="value")


def check_wells(values, defn):
    """Every well of the table has to exist in the plate definition."""
    unknown = [w for w in values.index if w not in set(defn.wells)]
    if unknown:
        raise ValueError(f"not wells of {defn.load_name!r}: {unknown[:10]}"
                         f"{' ...' if len(unknown) > 10 else ''}")
    return values


_NAME = re.compile(r"^([A-Za-z]+)(\d+)$")


def _cell(well):
    m = _NAME.match(well)
    return (m.group(1), int(m.group(2))) if m else (well, 0)


def plate_grid(defn, values, fill="") -> pd.DataFrame:
    """A table shaped like the plate. Row and column labels come from the
    definition's ordering rather than being generated, so custom plates are
    drawn correctly too."""
    rows, cols = [], []
    for column in defn.ordering:
        for well in column:
            r, c = _cell(well)
            if r not in rows:
                rows.append(r)
            if c not in cols:
                cols.append(c)
    grid = pd.DataFrame(fill, index=rows, columns=cols, dtype=object)
    for well, value in dict(values).items():
        r, c = _cell(well)
        if r in grid.index and c in grid.columns:
            grid.at[r, c] = value
    return grid


def wells_in_order(defn, how="by_row") -> list:
    """The plate's wells row-major or column-major, taken from its ordering."""
    if how == "by_column":
        return [w for column in defn.ordering for w in column]
    if how == "by_row":
        depth = max((len(c) for c in defn.ordering), default=0)
        return [column[r] for r in range(depth)
                for column in defn.ordering if r < len(column)]
    raise ValueError(f"how: 'by_row' or 'by_column', got {how!r}")

In [ ]:
# --- The file -------------------------------------------------------------
VALUES_FILE = paths.root() / "data" / "sort_values_3.xlsx"
LAYOUT      = "grid"       # "tecan" | "grid" | "long"
SHEET       = 0          # tecan: None = every sheet, averaged; or a name,
                            # or a list of names. grid/long: a name or an index.
AGG         = "mean"        # how repeated reads are combined: mean|median|first

values = check_wells(read_values(VALUES_FILE, LAYOUT, SHEET, agg=AGG), src_def)
print(f"{len(values)} wells carry a value, "
      f"from {values.min():.4g} to {values.max():.4g}")
plate_grid(src_def, values.round(4))

## 6. The assignment

Cuboids are sorted by value and laid into the destination in its own fill
order. `ORDER = "ascending"` with `DEST_ORDER = "by_row"` gives a gradient
running left to right across the destination's rows.

**When the source and the destination are the same plate,** wells that hold
anything are removed from the candidate destinations — every well in the table
of values, not only the selected ones, since the filtered-out wells hold
cuboids too. Any remaining overlap raises: depositing into a well still waiting
to be picked from scrambles the experiment beyond repair, so it is not a
warning.

In [ ]:
def build_mapping(values, src_def, dest_def, *, order="ascending",
                  dest_order="by_row", dest_start=None, exclude=(),
                  vmin=None, vmax=None, n_max=None, same_plate=False,
                  seed=0) -> pd.DataFrame:
    """Turn per-well values into a transfer plan: source well -> destination well.

    `values` is the whole table; the filters are applied in here, because on a
    shared plate occupancy is judged from the table as read, not from what
    survives the filters.
    """
    occupied = set(values.index)
    picked = values.drop(labels=[w for w in exclude if w in values.index])
    if vmin is not None:
        picked = picked[picked >= vmin]
    if vmax is not None:
        picked = picked[picked <= vmax]
    if picked.empty:
        raise ValueError("the filters left no wells at all")

    if order == "ascending":
        picked = picked.sort_values(kind="mergesort")
    elif order == "descending":
        picked = picked.sort_values(ascending=False, kind="mergesort")
    elif order == "as_is":
        pass
    elif order == "random":
        picked = picked.sample(frac=1.0, random_state=seed)
    else:
        raise ValueError(f"order: ascending|descending|as_is|random, "
                         f"got {order!r}")
    if n_max is not None:
        picked = picked.iloc[:int(n_max)]

    if isinstance(dest_order, (list, tuple)):
        dest_wells = [str(w).strip().upper() for w in dest_order]
        unknown = [w for w in dest_wells if w not in set(dest_def.wells)]
        if unknown:
            raise ValueError(f"not wells of {dest_def.load_name!r}: {unknown[:10]}")
    else:
        dest_wells = wells_in_order(dest_def, dest_order)

    if dest_start is not None:
        if dest_start not in dest_wells:
            raise ValueError(f"{dest_start!r} is not in the fill order")
        dest_wells = dest_wells[dest_wells.index(dest_start):]

    if same_plate:
        before = len(dest_wells)
        dest_wells = [w for w in dest_wells if w not in occupied]
        print(f"one plate: {before - len(dest_wells)} occupied wells excluded")

    if len(set(dest_wells)) != len(dest_wells):
        raise ValueError("the destination list repeats a well")
    if len(dest_wells) < len(picked):
        raise ValueError(f"{len(picked)} destination wells needed, "
                         f"{len(dest_wells)} available")

    plan = pd.DataFrame({
        "source_well": list(picked.index),
        "value": [float(v) for v in picked.values],
        "dest_well": dest_wells[:len(picked)],
    })
    plan.index = range(1, len(plan) + 1)
    plan.index.name = "n"

    if same_plate:
        clash = set(plan["dest_well"]) & occupied
        if clash:
            raise ValueError(f"destinations overlap occupied wells: "
                             f"{sorted(clash)[:10]}")
    return plan

In [ ]:
POOL_ROWS = "JKLMNO"
POOL_COLS = range(2, 12)        # поставьте те колонки, которые реально заполнены
POOL_WELL = "A1"

source_wells = [f"{r}{c}" for r in POOL_ROWS for c in POOL_COLS]
unknown = [w for w in source_wells if w not in set(src_def.wells)]
if unknown:
    raise ValueError(f"not wells of {src_def.load_name!r}: {unknown}")

order = [w for w in wells_in_order(src_def, "by_row") if w in set(source_wells)]
mapping = pd.DataFrame({"source_well": order,
                        "value": "",
                        "dest_well": POOL_WELL})
mapping.index = range(1, len(mapping) + 1)
mapping.index.name = "n"

print(f"{len(mapping)} wells -> {POOL_WELL}")
plate_grid(src_def, {w: "x" for w in source_wells})

In [ ]:
# --- Sorting rules --------------------------------------------------------
ORDER      = "ascending"    # "ascending" | "descending" | "as_is" | "random"
DEST_ORDER = [f"{r}{c}" for r in "JKLMNO" for c in range(2, 12)]       # "by_row" | "by_column" | an explicit list of wells
DEST_START = None           # first destination well; None for the very first
EXCLUDE    = []             # source wells to leave alone
VMIN, VMAX = None, None     # value thresholds
N_MAX      = None           # keep only the first N after sorting
SEED       = 0              # for ORDER = "random"

mapping = build_mapping(
    values, src_def, dest_def,
    order=ORDER, dest_order=DEST_ORDER, dest_start=DEST_START,
    exclude=EXCLUDE, vmin=VMIN, vmax=VMAX, n_max=N_MAX,
    same_plate=(SOURCE_SLOT == DEST_SLOT), seed=SEED)

print(f"{len(mapping)} transfers")
mapping

In [ ]:
# How the destination should end up: the value of the cuboid in each well.
plate_grid(dest_def,
           dict(zip(mapping["dest_well"], mapping["value"].round(4))))

In [ ]:
# And where each one came from.
plate_grid(dest_def, dict(zip(mapping["dest_well"], mapping["source_well"])))

## 7. The run

One cycle:

1. the robot parks above the source well, descends to `bottom_z() +
   ASPIRATE_Z` and aspirates `VOLUME`;
2. it lifts in small steps and holds above the well;
3. **it waits for you** — look at the tip and answer:

   | key | effect |
   |---|---|
   | `Enter` or `c` | a cuboid is in the tip — carry it to the destination |
   | `r` | nothing there — return the volume and try the same well again |
   | `s` | not working — return the volume and move to the next candidate |
   | `b` | return the volume and stop the run |

Offsets and heights are read from `setup` as each move is made, so the numbers
measured in sections 3 and 4 are the numbers used here, with nothing to
transcribe.

Every event is appended to the log immediately rather than at the end, so a run
stopped by an interrupt or a dead kernel still leaves a record of what happened.
Re-running the cell continues from the first unfinished well, as long as
`RUN_NAME` is unchanged.

Z is retracted in `finally`, including on `KeyboardInterrupt`. If there was
liquid in the tip at that moment the notebook says so — the cuboid stays in the
pipette and has to be dispensed by hand.

In [ ]:
# --- Motion parameters ----------------------------------------------------
# Tuning constants, not measurements: measured values live in `setup`.
HOVER_Z    = 0.5            # mm above the well top — the pose you inspect
ASPIRATE_Z = 0.0            # mm above the well bottom — the pickup height
LIFT_STEP  = 0.1            # small lift after aspirating
LIFT_STEPS = 5
RETURN_Z   = 2.0            # mm above the source bottom when giving it back
DEPOSIT_Z  = 2.0            # mm above the destination bottom when depositing

VOLUME        = 30.0        # ul
FLOW_RATE     = 200.0       # ul/s aspirating
DISPENSE_FLOW = 50.0       # ul/s dispensing
SETTLE        = 0.75        # pause after a dispense, s

show_setup()
print(f"\npickup at {bottom_z() + ASPIRATE_Z:.2f} mm, {VOLUME} ul")

In [ ]:
# --- The log --------------------------------------------------------------
RUN_NAME    = f"sort_{time.strftime('%Y%m%d_%H%M%S')}"
LOG_PATH    = paths.logs_dir() / f"{RUN_NAME}.csv"
RESULT_PATH = paths.outputs_dir() / f"{RUN_NAME}_mapping.csv"

LOG_FIELDS = ["time", "n", "source_well", "value", "dest_well", "attempt",
              "status", "volume", "note"]


def log_event(**row):
    """Append one row. The header is written when the file is created."""
    fresh = not LOG_PATH.exists()
    with open(LOG_PATH, "a", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=LOG_FIELDS)
        if fresh:
            writer.writeheader()
        writer.writerow({"time": time.strftime("%Y-%m-%d %H:%M:%S"),
                         **{k: row.get(k, "") for k in LOG_FIELDS
                            if k != "time"}})


def done_wells(status=("placed", "skipped")) -> set:
    """Source wells this log has already finished with."""
    if not LOG_PATH.exists():
        return set()
    log = pd.read_csv(LOG_PATH)
    return set(log.loc[log["status"].isin(status), "source_well"])


mapping.to_csv(RESULT_PATH.with_name(f"{RUN_NAME}_plan.csv"))
json.dump(setup, open(paths.outputs_dir() / f"{RUN_NAME}_setup.json", "w"),
          indent=2)
print("log:", LOG_PATH)

In [ ]:
# --- Elementary moves -----------------------------------------------------
# Each one reads the offsets and the bottom from `setup` at the moment it runs.

def aspirate_from(well):
    """Descend into a source well, aspirate, lift and hold above it."""
    offset = source_offset()
    bottom = bottom_z()
    well_top("source", well, offset=offset, z=HOVER_Z)
    x, y, _ = xyz(openapi)
    move_to(openapi, (x, y, bottom + ASPIRATE_Z),
            min_z_height=bottom, force_direct=True)
    require_ok(openapi.aspirate_in_place(volume=VOLUME, flow_rate=FLOW_RATE),
               "aspirate")
    for _ in range(LIFT_STEPS):
        move_relative(openapi, "z", LIFT_STEP)
    require_ok(openapi.move_to_well(src_lw, well, well_location="top",
                                    offset=(offset[0], offset[1], HOVER_Z),
                                    force_direct=True),
               "hover over source")


def return_to(well):
    """Give the volume back to the source well."""
    offset = source_offset()
    require_ok(openapi.dispense(src_lw, well, well_location="bottom",
                                offset=(offset[0], offset[1], RETURN_Z),
                                volume=VOLUME, flow_rate=DISPENSE_FLOW),
               "dispense back")
    time.sleep(SETTLE)
    require_ok(openapi.move_to_well(src_lw, well, well_location="top",
                                    offset=(offset[0], offset[1], HOVER_Z)),
               "retract from source")


def deposit_into(well):
    """Carry to the destination well and dispense."""
    offset = dest_offset()
    require_ok(openapi.move_to_well(dest_lw, well, well_location="top",
                                    offset=(offset[0], offset[1], HOVER_Z)),
               "move to destination")
    require_ok(openapi.dispense(dest_lw, well, well_location="bottom",
                                offset=(offset[0], offset[1], DEPOSIT_Z),
                                volume=VOLUME, flow_rate=DISPENSE_FLOW),
               "dispense to destination")
    time.sleep(SETTLE)
    require_ok(openapi.move_to_well(dest_lw, well, well_location="top",
                                    offset=(offset[0], offset[1], HOVER_Z)),
               "retract from destination")


def ask(n, total, source, dest, attempt) -> str:
    """Ask the operator, and keep asking until the answer parses."""
    prompt = (f"[{n}/{total}] {source} -> {dest}, attempt {attempt}  |  "
              f"Enter=picked  r=retry  s=skip  b=stop: ")
    while True:
        answer = input(prompt).strip().lower()
        if answer in ("", "c"):
            return "c"
        if answer in ("r", "p"):
            return "r"
        if answer == "s":
            return "s"
        if answer == "b":
            return "b"
        print("  not understood; use Enter, r, s or b")

In [ ]:
# 3. Новый прогон целиком, с новым именем
RUN_NAME = f"sort_{time.strftime('%Y%m%d_%H%M%S')}"
LOG_PATH = paths.logs_dir() / f"{RUN_NAME}.csv"

In [ ]:
# --- Run ------------------------------------------------------------------
plan = mapping.reset_index().to_dict("records")
completed = done_wells()
attempts = {}
holding = False

if completed:
    print(f"{len(completed)} wells already finished in this log; skipping\n")

openapi.retract_axis("leftZ")
idx = 0
try:
    while idx < len(plan):
        row = plan[idx]
        source, dest = row["source_well"], row["dest_well"]
        if source in completed:
            idx += 1
            continue

        attempts[source] = attempts.get(source, 0) + 1
        attempt = attempts[source]

        aspirate_from(source)
        holding = True
        action = ask(row["n"], len(plan), source, dest, attempt)

        common = dict(n=row["n"], source_well=source, value=row["value"],
                      dest_well=dest, attempt=attempt, volume=VOLUME)

        if action == "c":
            deposit_into(dest)
            holding = False
            log_event(status="placed", **common)
            print(f"  {source} -> {dest}")
            completed.add(source)
            idx += 1
        elif action == "r":
            return_to(source)
            holding = False
            log_event(status="retry", note="empty tip, retrying", **common)
            print(f"  {source}: returned, trying again")
        elif action == "s":
            return_to(source)
            holding = False
            log_event(status="skipped", note="skipped by operator", **common)
            print(f"  {source}: skipped")
            completed.add(source)
            idx += 1
        else:
            return_to(source)
            holding = False
            log_event(status="aborted", note="stopped by operator", **common)
            print("stopped by operator")
            break
finally:
    if holding:
        print("!! liquid is still in the tip — dispense it by hand")
        log_event(status="interrupted", note="interrupted holding liquid")
    openapi.retract_axis("leftZ")
    print("Z retracted")

## 8. Result

The log holds every attempt, retries included; the result table holds only what
arrived. It is written to `outputs/`, next to the plan and a copy of the setup
the run used.

In [ ]:
log = pd.read_csv(LOG_PATH)
print(log["status"].value_counts().to_string(), "\n")

placed = (log[log["status"] == "placed"]
          .drop_duplicates("source_well", keep="last")
          .set_index("n")[["source_well", "value", "dest_well", "attempt"]]
          .sort_index())
placed.to_csv(RESULT_PATH)
print("written:", RESULT_PATH)
placed

In [ ]:
# The destination as it actually is: the value now sitting in each filled well.
plate_grid(dest_def, dict(zip(placed["dest_well"], placed["value"].round(4))))

In [ ]:
# What did not make it, and why.
missed = sorted(set(mapping["source_well"]) - set(placed["source_well"]))
print(f"not transferred: {len(missed)}")
if missed:
    print(log[log["source_well"].isin(missed)]
          [["source_well", "dest_well", "attempt", "status", "note"]]
          .to_string(index=False))

In [ ]:
openapi.toggle_lights()